In [1]:
# import torch
# from torch_geometric.data import Data, Batch
# from torch_geometric.data import DataLoader as PyGDataLoader

# from concdvae.pl_data.dataset import CrystDataset
# from concdvae.common.data_utils import get_scaler_from_data_list
# from concdvae.common.data_utils import build_crystal_graph
# from eval_utils import load_model

In [2]:
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec

In [3]:
import torch
from torch_geometric.data import DataLoader as PyGDataLoader
from concdvae.pl_data.dataset import CrystDataset
from eval_utils import load_model
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
# from concdvae.common.data_utils import get_scaler_from_data_list
# from concdvae.common.data_utils import build_crystal_graph
# import inspect
# print(inspect.getsource(model.forward))

/afs/inf.ed.ac.uk/user/s23/s2305255/miniconda3/envs/concdvae310/lib/python3.10/site-packages/pyxtal/symmetry.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename as rf
/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/concdvae/pl_data/dataset.py:158: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path=str(PROJECT_ROOT / "conf"), config_name="default")
/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/concdvae/pl_data/datamodule.py:135: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path=

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
model_dir  = '/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/src/model/mp20_format/'
ckpt_name  = 'epoch=330-step=17543.ckpt'

model, test_loader, cfg = load_model(model_path=model_dir, model_file=ckpt_name)
model.to(device)
model.eval()

load ckpt: /afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/src/model/mp20_format/epoch=330-step=17543.ckpt


/afs/inf.ed.ac.uk/user/s23/s2305255/miniconda3/envs/concdvae310/lib/python3.10/site-packages/hydra/experimental/initialize.py:116: UserWarning: hydra.experimental.initialize_config_dir() is no longer experimental. Use hydra.initialize_config_dir().
  deprecation_warning(message=message)
/afs/inf.ed.ac.uk/user/s23/s2305255/miniconda3/envs/concdvae310/lib/python3.10/site-packages/hydra/experimental/initialize.py:118: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  self.delegate = real_initialize_config_dir(
/afs/inf.ed.ac.uk/user/s23/s2305255/miniconda3/envs/concdvae310/lib/python3.10/site-packages/hydra/experimental/compose.py:25: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  deprecation_warning(message=message)
/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/scripts/eval_utils.py:76: FutureWarning: You are using `torch.load` wit

CDVAE(
  (conditions_predict): ModuleList(
    (0): ScalarConditionPredict(
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Linear(in_features=256, out_features=256, bias=True)
        (4): ReLU()
        (5): Linear(in_features=256, out_features=1, bias=True)
      )
    )
  )
  (condition_model): ConditioningModule(
    (condition_embModel): ModuleList(
      (0): ScalarConditionEmbedding(
        (gaussian_expansion): GaussianRBF()
        (dense_net): Sequential(
          (0): Linear(in_features=15, out_features=64, bias=True)
          (1): ReLU()
          (2): Linear(in_features=64, out_features=64, bias=True)
          (3): ReLU()
          (4): Linear(in_features=64, out_features=64, bias=True)
          (5): ReLU()
          (6): Linear(in_features=64, out_features=64, bias=True)
        )
      )
    )
    

In [6]:
# print(type(model))
# print(type(test_loader))
# print(type(cfg))

In [7]:
dataset = CrystDataset(
    name='Formation energy test',
    path='/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/data/mp_20/test.csv',
    prop=cfg.data.prop,
    niggli=cfg.data.niggli,
    primitive=cfg.data.primitive,
    graph_method=cfg.data.graph_method,
    lattice_scale_method=cfg.data.lattice_scale_method,
    preprocess_workers=4,
    save_path='/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/data/mp_20/test_data.pt',
    tolerance=cfg.data.tolerance,
    use_space_group=cfg.data.use_space_group,
    use_pos_index=cfg.data.use_pos_index,
    load_old=True,   # ← flip to False on first run to preprocess
    prelo_prop=cfg.data.prelo_prop,
)

loader = PyGDataLoader(dataset, batch_size=8, shuffle=False, num_workers=0)
batch  = next(iter(loader)).to(device)

print(f"Batch: {batch.num_atoms.shape[0]} crystals, {batch.num_atoms.sum().item()} atoms")
print(f"Formation energies: {batch.formation_energy_per_atom.cpu().numpy().round(3)}")


/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/concdvae/pl_data/dataset.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.cached_data = torch.load(save_pa

Batch: 8 crystals, 74 atoms
Formation energies: [-0.575 -0.942  0.065 -1.456  0.024 -2.117 -1.361 -0.139]


/afs/inf.ed.ac.uk/user/s23/s2305255/miniconda3/envs/concdvae310/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [8]:
for name, module in model.named_modules():
    module._forward_hooks.clear()

activations = {}

def capture(name):
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            activations[name] = output.detach().cpu()
        elif isinstance(output, tuple):
            activations[name] = tuple(
                o.detach().cpu() if isinstance(o, torch.Tensor) else o
                for o in output
            )
    return hook

hooks = [
    model.fc_mu.register_forward_hook(capture('z_mu')),
    model.fc_var.register_forward_hook(capture('z_var')),
    model.condition_model.register_forward_hook(capture('cond_emb')),
    model.z_condition.register_forward_hook(capture('z_cond')),
]

In [9]:
with torch.no_grad():
    model(batch, teacher_forcing=False, training=False)

for h in hooks:
    h.remove()

/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/concdvae/pl_modules/gnn.py:393: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at ../aten/src/ATen/native/Cross.cpp:62.)
  b = torch.cross(pos_ji, pos_kj).norm(dim=-1)
/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/concdvae/common/data_utils.py:690: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/concdvae/common/data_utils.py:686: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().

In [10]:
z_mu    = activations['z_mu']                          # (8, 256)
z_var   = activations['z_var']                         # (8, 256)
z       = z_mu + torch.randn_like(z_mu) * (0.5 * z_var).exp()  # (8, 256)
cond_emb = activations['cond_emb']                     # (8, 128)
z_cond  = activations['z_cond']                        # (8, 256)

record = {
    'z_mu':            z_mu,
    'z_var':           z_var,
    'z':               z,
    'cond_emb':        cond_emb,
    'z_cond':          z_cond,
    'formation_energy': batch.formation_energy_per_atom.cpu(),
    'composition':     [dataset[i].atom_types for i in range(8)],
}


In [11]:
for k, v in record.items():
    if isinstance(v, torch.Tensor):
        print(f"{k:20s} {v.shape}")
    else:
        print(f"{k:20s} {type(v)}")

z_mu                 torch.Size([8, 256])
z_var                torch.Size([8, 256])
z                    torch.Size([8, 256])
cond_emb             torch.Size([8, 128])
z_cond               torch.Size([8, 256])
formation_energy     torch.Size([8])
composition          <class 'list'>


In [15]:
BASE = '/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/data/mp_20'

all_records = {
    'z_mu':             [],
    'z_var':            [],
    'z':                [],
    'cond_emb':         [],
    'z_cond':           [],
    'formation_energy': [],
}

loader_full = PyGDataLoader(dataset, batch_size=8, shuffle=False, num_workers=0)

for batch in loader_full:
    batch = batch.to(device)

    activations = {}
    hooks = [
        model.fc_mu.register_forward_hook(capture('z_mu')),
        model.fc_var.register_forward_hook(capture('z_var')),
        model.condition_model.register_forward_hook(capture('cond_emb')),
        model.z_condition.register_forward_hook(capture('z_cond')),
    ]

    with torch.no_grad():
        model(batch, teacher_forcing=False, training=False)

    for h in hooks:
        h.remove()

    z_mu = activations['z_mu']
    z_var = activations['z_var']
    z     = z_mu + torch.randn_like(z_mu) * (0.5 * z_var).exp()

    all_records['z_mu'].append(z_mu)
    all_records['z_var'].append(z_var)
    all_records['z'].append(z)
    all_records['cond_emb'].append(activations['cond_emb'])
    all_records['z_cond'].append(activations['z_cond'])
    all_records['formation_energy'].append(batch.formation_energy_per_atom.cpu())

for key in ['z_mu', 'z_var', 'z', 'cond_emb', 'z_cond', 'formation_energy']:
    all_records[key] = torch.cat(all_records[key], dim=0)

print(f"Total crystals: {all_records['z_cond'].shape[0]}")
for k, v in all_records.items():
    if isinstance(v, torch.Tensor):
        print(f"{k:20s} {v.shape}")

torch.save(all_records, f'{BASE}/activation_dataset.pt')
print(f"Saved → {BASE}/activation_dataset.pt")

Total crystals: 9046
z_mu                 torch.Size([9046, 256])
z_var                torch.Size([9046, 256])
z                    torch.Size([9046, 256])
cond_emb             torch.Size([9046, 128])
z_cond               torch.Size([9046, 256])
formation_energy     torch.Size([9046])
Saved → /afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/data/mp_20/activation_dataset.pt


In [13]:
data = torch.load('/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/data/mp_20/activation_dataset.pt')

/tmp/ipykernel_167974/3364570949.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load('/afs/inf.ed.ac.uk/user/s23/s2305255/Desktop/MLP/Con-CDVAE/data/mp_20/

In [14]:
# Check for NaNs or Infs in any vector
for k in ['z_mu', 'z_var', 'z', 'cond_emb', 'z_cond']:
    n_nan = data[k].isnan().sum().item()
    n_inf = data[k].isinf().sum().item()
    print(f"{k:20s}  NaN: {n_nan}  Inf: {n_inf}  mean: {data[k].mean():.4f}  std: {data[k].std():.4f}")

# Check formation energy range looks sensible
fe = data['formation_energy']
print(f"\nFormation energy  min: {fe.min():.3f}  max: {fe.max():.3f}  mean: {fe.mean():.3f}")

z_mu                  NaN: 0  Inf: 0  mean: 0.0005  std: 0.3014
z_var                 NaN: 0  Inf: 0  mean: -0.3005  std: 0.9911
z                     NaN: 0  Inf: 0  mean: 0.0009  std: 1.0003
cond_emb              NaN: 0  Inf: 0  mean: 0.0126  std: 0.1953
z_cond                NaN: 0  Inf: 0  mean: -0.0029  std: 0.6192

Formation energy  min: -4.472  max: 0.080  mean: -1.211
